# パターン認識特論 第2回 演習：ニューラルネットワークの基礎アーキテクチャ

講義スライド（Version 7）の式を，そのまま NumPy で動かして確かめる．
各節の最後に **やってみよう** がある．`# TODO` の部分を書き換えて実行する．解答は直後の「解答」セルにある（先に自分で試す）．

| 節 | テーマ | スライド |
|---|---|---|
| §1 | ユニットの計算 | p.4 |
| §2 | 活性化関数と微分 | p.6–8 |
| §3 | MLP の順伝播と表現変換 | p.9–11 |
| §4 | ソフトマックスと交差エントロピー | p.13 |
| §5 | 誤差信号 $\boldsymbol{p}-\boldsymbol{y}$ の検証 | p.14 |
| §6 | 埋め込み | p.15 |
| §7 | （任意）PyTorch で同じことをする | 付録B |

環境：Python 3 ＋ NumPy ＋ Matplotlib．§7 のみ PyTorch（入っていなければ読み飛ばす）．

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## §1 ユニットの計算（p.4）

$$u = \boldsymbol{w}^\top \boldsymbol{x} + b, \qquad z = h(u)$$

スライドの数値例：$\boldsymbol{x}=(2,1)^\top$，$\boldsymbol{w}=(0.5,-1)^\top$，$b=0.2$，$h=\mathrm{ReLU}$．

In [ ]:
def relu(u):
    return np.maximum(0.0, u)

x = np.array([2.0, 1.0])
w = np.array([0.5, -1.0])
b = 0.2
u = w @ x + b          # 重み付き和（内積）＋バイアス
z = relu(u)            # 活性化関数
print(f"u = {u:.3f}, z = ReLU(u) = {z:.3f}")   # スライドと同じ 0.2, 0.2 になるはず

**やってみよう 1-1**　$x_2$ を 1 から 2 に変えると $u$ と $z$ はどうなるか．先に予想してから実行する
（$w_2=-1$ なので，$x_2$ を増やすと総入力は**下がる**）．

In [ ]:
x2 = np.array([2.0, 1.0])   # TODO: x2[1] を 2.0 に変える
u2 = w @ x2 + b
print(f"u = {u2:.3f}, z = {relu(u2):.3f}")

In [ ]:
# 解答
x2 = np.array([2.0, 2.0])
u2 = w @ x2 + b
print(f"u = {u2:.3f}, z = {relu(u2):.3f}")   # u = -0.8 → ReLU で z = 0（このユニットは「発火しない」）

## §2 活性化関数と微分（p.6–8）

シグモイド・tanh・ReLU・GELU と，その微分を式どおりに実装して描く．

In [ ]:
from math import erf, sqrt

def sigmoid(u):
    return 1.0 / (1.0 + np.exp(-u))

def d_sigmoid(u):
    s = sigmoid(u)
    return s * (1.0 - s)

def d_tanh(u):
    return 1.0 - np.tanh(u) ** 2

def d_relu(u):
    return (u > 0).astype(float)

Phi = np.vectorize(lambda t: 0.5 * (1.0 + erf(t / sqrt(2.0))))   # 標準正規分布の累積分布関数

def gelu(u):
    return u * Phi(u)

us = np.linspace(-4, 4, 401)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(us, sigmoid(us), ':', label='sigmoid')
ax[0].plot(us, np.tanh(us), '-.', label='tanh')
ax[0].plot(us, relu(us), '-', label='ReLU')
ax[0].plot(us, gelu(us), '--', label='GELU')
ax[0].set_title('h(u)'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(us, d_sigmoid(us), ':', label="sigmoid'")
ax[1].plot(us, d_tanh(us), '-.', label="tanh'")
ax[1].plot(us, d_relu(us), '-', label="ReLU'")
ax[1].set_title("h'(u)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("sigmoid'(u) の最大値 =", d_sigmoid(0.0), "（u=0 で 0.25）")
print("0.25 を 10 層分掛けると =", 0.25 ** 10)

**やってみよう 2-1**　シグモイドの微分を 20 層分，50 層分掛けるといくらになるか．また，ReLU の正側（微分 1）なら何層でも 1 のままであることを確かめる．

In [ ]:
for n_layers in (10, 20, 50):
    print(f"{n_layers:2d} 層: sigmoid 側 {0.25 ** n_layers:.2e}, ReLU 側 {1.0 ** n_layers:.1f}")

**やってみよう 2-2（dying ReLU，p.8 の小話）**　ReLU の隠れ層 1 つのネットワークで，「どの入力に対しても出力が 0 のユニット」（死んだユニット）を数える．
学習率が大きすぎて更新が飛ぶと，バイアスが大きく負に振れたユニットが生まれる．ここではその状態を，バイアスを総入力の標準偏差の `shift` 倍だけ負に寄せて模擬する．
`shift` を 0 → 4 と変えると，死んだユニットが増えていく．

In [ ]:
def count_dead_units(shift, n_hidden=256, n_samples=1000, d_in=10):
    X = rng.normal(size=(n_samples, d_in))
    W = rng.normal(size=(n_hidden, d_in)) / np.sqrt(d_in)
    U0 = X @ W.T
    b = -shift * U0.std(axis=0)                            # バイアスを負に寄せる（飛んだ更新の模擬）
    Z = relu(U0 + b)                                       # (n_samples, n_hidden)
    dead = np.all(Z == 0, axis=0)                          # 全サンプルで 0 のユニット
    return int(dead.sum())

for shift in (0.0, 2.0, 3.0, 4.0):     # TODO: 値を変えて試す
    print(f"shift={shift}: 死んだユニット {count_dead_units(shift)} / 256")

## §3 MLP の順伝播と表現変換（p.9–11）

$$\boldsymbol{z}^{(l)} = h\left(W^{(l)}\boldsymbol{z}^{(l-1)} + \boldsymbol{b}^{(l)}\right),\qquad \boldsymbol{z}^{(0)}=\boldsymbol{x}$$

p.9 の 2–3–2 の MLP を，小さな数値で手計算と一致させる．

In [ ]:
W1 = np.array([[1.0, -1.0],
               [0.5,  0.5],
               [-1.0, 2.0]])          # 3×2：第1層（入力2 → 隠れ3）
b1 = np.array([0.0, 0.1, -0.5])
W2 = np.array([[1.0, 0.0, -1.0],
               [0.0, 2.0,  1.0]])     # 2×3：第2層（隠れ3 → 出力2）
b2 = np.array([0.0, 0.0])

x = np.array([1.0, 2.0])
u1 = W1 @ x + b1
z1 = relu(u1)
u2 = W2 @ z1 + b2                     # 出力層は恒等（回帰）とする
print("u1 =", u1, " z1 =", z1)
print("u2 =", u2)

n_params = W1.size + b1.size + W2.size + b2.size
print("パラメータ数 =", n_params, "（重み 6+6=12，バイアス 3+2=5）")

**やってみよう 3-1（p.10 のクイズ）**　784 → 256 → 10 の MLP のパラメータ数を式 $N^{(l)}N^{(l-1)}+N^{(l)}$ で計算し，203,530 になることを確かめる．
次に 224×224×3 = 150,528 次元の入力で隠れ層 256 なら第1層だけで何個になるか．

In [ ]:
def n_params_mlp(sizes):
    return sum(sizes[l] * sizes[l - 1] + sizes[l] for l in range(1, len(sizes)))

print("784-256-10:", n_params_mlp([784, 256, 10]))
print("150528-256 の第1層:", n_params_mlp([150528, 256]))   # TODO: 隠れ層を 1024 にすると？

### 表現変換を見る（p.11）

同心円の 2 クラスは入力空間では直線で分けられない．隠れ層 2 ユニットの MLP（2 → 8 → 2 → 1）を訓練し，
最後の隠れ層の出力（2 次元）を散布図にすると，**直線で分けられる形**に写されていることが見える．

訓練ループの中身（逆伝播）は第3回で学ぶ．ここでは「動かして結果を見る」だけでよい．

In [ ]:
# データ：内側の円（クラス0）と外側の輪（クラス1）
n = 200
r = np.concatenate([rng.uniform(0.0, 1.0, n), rng.uniform(1.6, 2.6, n)])
th = rng.uniform(0, 2 * np.pi, 2 * n)
X = np.stack([r * np.cos(th), r * np.sin(th)], axis=1)
y = np.concatenate([np.zeros(n), np.ones(n)])

# 2 → 8 → 2 → 1 の MLP（隠れ層 tanh，出力 シグモイド，二値交差エントロピー）
def init(shape, scale=1.0):
    return rng.normal(size=shape) * scale / np.sqrt(shape[1])

P = dict(W1=init((8, 2)), b1=np.zeros(8), W2=init((2, 8)), b2=np.zeros(2), W3=init((1, 2)), b3=np.zeros(1))

def forward(P, X):
    u1 = X @ P['W1'].T + P['b1']; z1 = np.tanh(u1)
    u2 = z1 @ P['W2'].T + P['b2']; z2 = np.tanh(u2)
    u3 = z2 @ P['W3'].T + P['b3']; p = sigmoid(u3[:, 0])
    return p, (X, u1, z1, u2, z2)

def train_step(P, X, y, lr=0.1):
    # ---- 逆伝播（中身は第3回）．δ^(L) = p − y から始めて入力側へ．
    p, (X, u1, z1, u2, z2) = forward(P, X)
    N = len(y)
    d3 = (p - y)[:, None] / N
    gW3 = d3.T @ z2; gb3 = d3.sum(0)
    d2 = (d3 @ P['W3']) * (1 - z2 ** 2)
    gW2 = d2.T @ z1; gb2 = d2.sum(0)
    d1 = (d2 @ P['W2']) * (1 - z1 ** 2)
    gW1 = d1.T @ X; gb1 = d1.sum(0)
    for k, g in zip(['W1', 'b1', 'W2', 'b2', 'W3', 'b3'], [gW1, gb1, gW2, gb2, gW3, gb3]):
        P[k] = P[k] - lr * g
    loss = -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
    return loss

losses = [train_step(P, X, y, lr=0.5) for _ in range(3000)]
p, (_, _, _, _, z2) = forward(P, X)
acc = np.mean((p > 0.5) == (y == 1))
print(f"最終損失 {losses[-1]:.3f}, 正解率 {acc:.3f}")

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].scatter(X[y == 0, 0], X[y == 0, 1], s=12, label='class 0'); ax[0].scatter(X[y == 1, 0], X[y == 1, 1], s=12, marker='^', label='class 1')
ax[0].set_title('(a) input space x'); ax[0].set_aspect('equal'); ax[0].legend()
ax[1].scatter(z2[y == 0, 0], z2[y == 0, 1], s=12); ax[1].scatter(z2[y == 1, 0], z2[y == 1, 1], s=12, marker='^')
# 出力層は z2 上の線形識別器： W3 z2 + b3 = 0 が境界
w3, b3 = P['W3'][0], P['b3'][0]
xs = np.linspace(-1.1, 1.1, 2)
if abs(w3[1]) > 1e-6:
    ax[1].plot(xs, -(w3[0] * xs + b3) / w3[1], 'k--', label='linear classifier')
ax[1].set_xlim(-1.1, 1.1); ax[1].set_ylim(-1.1, 1.1); ax[1].set_title('(b) representation space z^(2)'); ax[1].legend()
ax[2].plot(losses); ax[2].set_title('training loss'); ax[2].set_xlabel('step')
plt.tight_layout(); plt.show()

**やってみよう 3-2**　隠れ層の tanh を恒等写像（`np.tanh` を外す）に変えると何が起きるか予想し，試す（p.5：活性化関数がなければ何層重ねても 1 つのアフィン変換）．
上のセルの `forward` と `train_step` をコピーして `np.tanh(u)` → `u`，微分 `(1 - z**2)` → `1` に置き換える．

In [ ]:
# 解答（要点）：表現空間でも同心円のままで，直線では分けられず正解率は 0.5 前後にとどまる．
# 線形変換の合成は線形変換であり，「内側か外側か」という非線形な境界を作れないため．
print("自分で書き換えて確かめる．")

## §4 ソフトマックスと交差エントロピー（p.13）

$$p_k=\frac{\exp(u_k)}{\sum_j \exp(u_j)},\qquad L=-\log p_y$$

In [ ]:
def softmax(u):
    u = u - u.max()          # オーバーフロー対策（結果は変わらない．下で確かめる）
    e = np.exp(u)
    return e / e.sum()

def cross_entropy(u, y):
    return -np.log(softmax(u)[y])

u = np.array([0.0, 2.0])
print("p =", softmax(u))                       # (0.119, 0.881)
print("正解が 1 のとき L =", cross_entropy(u, 1))   # -log 0.881 = 0.127
print("正解が 0 のとき L =", cross_entropy(u, 0))   # -log 0.119 = 2.127

**やってみよう 4-1**　ロジット $\boldsymbol{u}$ の全成分に同じ定数を足しても $\boldsymbol{p}$ は変わらない（ソフトマックスは差だけを見る）．定数を変えて確かめる．
**やってみよう 4-2**　$p_y$ が 0.2, 0.8, 0.99, 1.0 のときの損失を計算し，p.13 の値と比べる．

In [ ]:
for c in (0.0, 5.0, -100.0):   # TODO: 別の値でも
    print(f"u + {c:6.1f}: p = {softmax(u + c)}")
for py in (0.2, 0.8, 0.99, 1.0):
    print(f"p_y = {py}: L = {-np.log(py):.3f}")

## §5 誤差信号 $\boldsymbol{\delta}^{(L)}=\boldsymbol{p}-\boldsymbol{y}$ の検証（p.14）

ソフトマックス＋交差エントロピーで，出力層の総入力（ロジット）に関する損失の勾配が本当に $\boldsymbol{p}-\boldsymbol{y}$ になるか，
**数値微分**（各成分を $\varepsilon$ だけ動かして損失の変化を測る）と比べる．

In [ ]:
def numerical_grad(f, u, eps=1e-5):
    g = np.zeros_like(u)
    for k in range(len(u)):
        e = np.zeros_like(u); e[k] = eps
        g[k] = (f(u + e) - f(u - e)) / (2 * eps)     # 中心差分
    return g

u = np.array([0.5, -1.0, 2.0, 0.0])
y = 2                                    # 正解クラス
y_onehot = np.eye(4)[y]
p = softmax(u)
print("p - y        =", p - y_onehot)
print("数値微分     =", numerical_grad(lambda v: cross_entropy(v, y), u))
print("最大誤差     =", np.abs(p - y_onehot - numerical_grad(lambda v: cross_entropy(v, y), u)).max())

**やってみよう 5-1**　二値分類（シグモイド＋二値交差エントロピー）でも $\partial L/\partial u = p - y$ になることを，同じ方法で確かめる．
**やってみよう 5-2**　「悪い対」の例：シグモイド出力に**二乗誤差** $L=\frac{1}{2}(p-y)^2$ を組み合わせると，$u=10$（飽和）のとき勾配がほぼ 0 になることを数値微分で見る．

In [ ]:
def bce(u, y):
    p = sigmoid(u)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

u0, y0 = np.array([1.5]), 1.0
print("二値：p - y =", sigmoid(u0) - y0, " 数値微分 =", numerical_grad(lambda v: bce(v[0], y0), u0))

def sq_err(u, y):
    return 0.5 * (sigmoid(u) - y) ** 2

for uu in (0.0, 10.0):   # 飽和すると勾配が消える
    g = numerical_grad(lambda v: sq_err(v[0], 0.0), np.array([uu]))
    print(f"シグモイド＋二乗誤差, u={uu:4.1f}, y=0: dL/du = {g[0]:.2e}")

## §6 埋め込み（p.15）

語彙 $V$ 個の記号に $d$ 次元ベクトルを割り当てる表 $E\in\mathbb{R}^{d\times V}$．記号 $i$ の 1-hot ベクトルに $E$ を掛けると第 $i$ 列 $\boldsymbol{e}_i$ が取り出される．
ここでは学習済みのふりをした小さな表を手で作り，コサイン類似度で近い記号を探す．

In [ ]:
vocab = ['りんご', 'みかん', 'ぶどう', 'ノートPC', 'スマホ', 'キーボード']
V, d = len(vocab), 4
E = np.array([[0.9, 0.1, 0.0, 0.2],     # りんご
              [0.8, 0.2, 0.1, 0.1],     # みかん
              [0.7, 0.3, 0.0, 0.3],     # ぶどう
              [0.0, 0.1, 0.9, 0.8],     # ノートPC
              [0.1, 0.0, 0.8, 0.9],     # スマホ
              [0.0, 0.2, 0.9, 0.6]]).T  # キーボード  → E は d×V
print("E の形 =", E.shape, "（d×V），パラメータ数 =", E.size)

i = vocab.index('ノートPC')
onehot = np.eye(V)[i]
print("1-hot × E^T =", E @ onehot, " ＝ 第 i 列 =", E[:, i])

def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

S = np.array([[cosine(E[:, a], E[:, b]) for b in range(V)] for a in range(V)])
print("\nコサイン類似度（行：クエリ）")
for a in range(V):
    order = np.argsort(-S[a])[1:3]
    print(f"{vocab[a]:8s} に近い: {vocab[order[0]]}({S[a, order[0]]:.2f}), {vocab[order[1]]}({S[a, order[1]]:.2f})")

**やってみよう 6-1**　`E` に新しい記号「バナナ」の列を追加し，果物の仲間に近くなるようなベクトルを自分で決めて，最近傍が果物になることを確かめる．
**やってみよう 6-2**　語彙 50,000・次元 768（BERT-base 相当）の埋め込み行列のパラメータ数はいくつか．

In [ ]:
print("50000 × 768 =", 50000 * 768)   # 約 3,840 万個（BERT-base の全パラメータ 1.1 億個のうち 1/3 が入口の表）

## §7（任意）PyTorch で同じことをする（付録B）

PyTorch が入っていれば，§3 の MLP・§4 の損失・§6 の埋め込みが `nn.Linear`，`nn.CrossEntropyLoss`，`nn.Embedding` にそのまま対応することを見る．

In [ ]:
try:
    import torch
    import torch.nn as nn
    torch.manual_seed(0)
    mlp = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
    print("パラメータ数 =", sum(p.numel() for p in mlp.parameters()))   # 203530（§3 と一致）

    # CrossEntropyLoss は「ロジット」を受け取る（確率にしてから渡すのは典型的なバグ）
    logits = torch.tensor([[0.0, 2.0]])
    target = torch.tensor([1])
    print("CE(logits) =", nn.CrossEntropyLoss()(logits, target).item())                    # 0.127（§4 と一致）
    print("CE(softmax(logits)) =", nn.CrossEntropyLoss()(torch.softmax(logits, 1), target).item(), "← 値がずれる")

    emb = nn.Embedding(num_embeddings=6, embedding_dim=4)
    print("Embedding の重みの形 =", tuple(emb.weight.shape), "（PyTorch は V×d で持つ）")
    print("記号 3 のベクトル =", emb(torch.tensor([3])).detach().numpy())
except ImportError:
    print("PyTorch が見つからないので，この節は読み飛ばす．")